# F1 Race Prediction — Exploratory Data Analysis
**Objective:** Understand class balance, feature distributions, driver/team trends, and qualifying-finish correlations across 2021–2025.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df = pd.read_parquet('../data/processed/model_table.parquet')
print(f'Shape: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Unique drivers: {df["driver_id"].nunique()}')
print(f'Unique constructors: {df["constructor_id"].nunique()}')
df.head()

In [ ]:
print('Null counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nData types:')
print(df.dtypes)

## Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, target, label in zip(
    axes,
    ['is_top3', 'is_winner'],
    ['Podium (Top 3)', 'Race Winner']
):
    counts = df[target].value_counts()
    ax.bar(['Not ' + label, label], [counts[0], counts[1]], color=['#4C72B0', '#DD8452'])
    ax.set_title(f'Class Balance — {label}')
    ax.set_ylabel('Count')
    positive_rate = counts[1] / counts.sum()
    ax.set_xlabel(f'Positive rate: {positive_rate:.1%}')

plt.tight_layout()
plt.savefig('../data/processed/eda_class_balance.png', dpi=120, bbox_inches='tight')
plt.show()

## Qualifying vs Race Finish

In [ ]:
if 'qualifying_position' in df.columns:
    sample = df.dropna(subset=['qualifying_position', 'finish_position'])
    plt.figure(figsize=(8, 6))
    plt.scatter(sample['qualifying_position'], sample['finish_position'],
                alpha=0.3, s=20, color='steelblue')
    corr = sample[['qualifying_position', 'finish_position']].corr().iloc[0, 1]
    plt.title(f'Qualifying Position vs Race Finish (Pearson r = {corr:.3f})')
    plt.xlabel('Qualifying Position')
    plt.ylabel('Race Finish Position')
    plt.savefig('../data/processed/eda_quali_vs_finish.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Correlation: {corr:.3f}')
else:
    print('qualifying_position not available in model table.')

## Top 10 Drivers by Average Finish Position (2021–2025)

In [ ]:
top_drivers = (
    df.groupby('driver_id')['finish_position']
    .mean()
    .sort_values()
    .head(10)
)

plt.figure(figsize=(10, 5))
colors = cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top_drivers)))
bars = plt.barh(top_drivers.index[::-1], top_drivers.values[::-1], color=colors[::-1])
plt.title('Top 10 Drivers by Average Finish Position (lower = better)')
plt.xlabel('Average Finish Position')
plt.tight_layout()
plt.savefig('../data/processed/eda_top_drivers.png', dpi=120, bbox_inches='tight')
plt.show()

## Podium Rate per Driver (Top 15)

In [ ]:
podium_rate = (
    df.groupby('driver_id')['is_top3']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'podium_rate', 'count': 'races'})
    .query('races >= 10')
    .sort_values('podium_rate', ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 5))
plt.bar(podium_rate.index, podium_rate['podium_rate'], color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.title('Podium Rate by Driver (min 10 races)')
plt.ylabel('Podium Rate')
plt.tight_layout()
plt.savefig('../data/processed/eda_podium_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## Feature Correlation Heatmap

In [ ]:
numeric_features = [
    'qualifying_position', 'grid', 'avg_finish_last5', 'std_finish_last5',
    'recent_form_last3', 'constructor_avg_pts_last5', 'track_avg_finish',
    'is_top3', 'is_winner', 'finish_position'
]
available = [c for c in numeric_features if c in df.columns]
corr_matrix = df[available].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## Rolling Form vs Podium Outcome

In [ ]:
if 'avg_finish_last5' in df.columns:
    plt.figure(figsize=(10, 5))
    for label, grp in df.groupby('is_top3'):
        vals = grp['avg_finish_last5'].dropna()
        plt.hist(vals, bins=20, alpha=0.6,
                 label='Podium' if label == 1 else 'Non-podium',
                 density=True)
    plt.legend()
    plt.title('Distribution of avg_finish_last5 by Podium Outcome')
    plt.xlabel('5-Race Rolling Average Finish Position')
    plt.ylabel('Density')
    plt.tight_layout()
    plt.savefig('../data/processed/eda_rolling_form_dist.png', dpi=120, bbox_inches='tight')
    plt.show()

## Constructor Points Trend by Season

In [ ]:
if 'constructor_id' in df.columns:
    top_constructors = (
        df.groupby('constructor_id')['points'].sum()
        .sort_values(ascending=False)
        .head(6)
        .index
    )
    season_pts = (
        df[df['constructor_id'].isin(top_constructors)]
        .groupby(['season', 'constructor_id'])['points']
        .sum()
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    for constructor in top_constructors:
        data = season_pts[season_pts['constructor_id'] == constructor]
        ax.plot(data['season'], data['points'], marker='o', label=constructor)
    ax.set_title('Top 6 Constructors — Total Points per Season')
    ax.set_xlabel('Season')
    ax.set_ylabel('Total Points')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/processed/eda_constructor_trend.png', dpi=120, bbox_inches='tight')
    plt.show()

## EDA Summary

| Finding | Value |
|---|---|
| Total driver-race rows | See shape above |
| is_top3 positive rate | ~15% (3 podiums per 20-driver field) |
| is_winner positive rate | ~5% (1 winner per 20-driver field) |
| Qualifying–finish correlation | Strong positive (r ≈ 0.4–0.6) |
| Strongest feature (expected) | qualifying_position, avg_finish_last5 |

**Class imbalance action:** Use `class_weight='balanced'` (LR) and `scale_pos_weight` (XGBoost).
